In [ ]:
try:
    # In Google Colab: %matplotlib widget needs ipympl installed and Colab's custom widget manager
    import google.colab
    %pip install -q ipympl
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
#import pandas as pd

# The widget backend keeps every figure alive: close the previous run's before drawing new ones
plt.close('all')

# A fixed seed: the same samples on every run
rng = np.random.default_rng(1)

sample_size = 360
mu, sigma = 0.0, 1.0

def equal_scales(ax, data):
    """The same range on all three axes, so a unit is equally long along each of them in a cubic box.

    Fitting each axis to its own data would rescale a cloud stretched along one axis and squeezed along
    another back into a round ball. The range is at least mu +/- 4 sigma, so the plots share it.
    """
    half = max(4 * sigma, np.abs(data - mu).max())
    for set_limits in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        set_limits(mu - half, mu + half)

independent_variables = rng.normal(mu, sigma, (3, sample_size))

desired_correlation = np.array(
        [[1.0, 0.95, -0.85],
        [0.95, 1.0, -0.9],
        [-0.85, -0.9, 1.0]])
print("Desired correlation:")
print(desired_correlation)

# X - generated data with E[X_i] = 0,
#                         stdev(X_i) = 1,
#                         cov(X_i, X_j) = 0  if i != j
#                         cov(X_i, X_j) = 1  if i = j
#
# Covariance matrix = correlation matrix = just an identity matrix because stdev(X_i) = 1
# C_xx = E[(X - E[X]) (X - E[X])^T] = E[X X^T] - E[X] E[X^T] = E[X X^T] = R_xx = I
#
# Y = A * X
# R_yy = E[Y * Y^T] = E[A X X^T A^T] = A E[X X^T] A^T = A A^T     <- desired correlation matrix
# R_yy = L L^* = L L^T = A A^T          <-- L is Cholesky decomposition of R_yy
# (L^* = L^T because L is a real matrix: the correlation matrix R_yy is real)
# A = L

L = np.linalg.cholesky(desired_correlation)
print("Cholesky decomposition of correlation matrix:")
print(L)

#correlated_variables = L.dot(independent_variables)
correlated_variables = L @ independent_variables

print("Actual correlation of new variables:")
actual_correlation = np.corrcoef(correlated_variables)
print(actual_correlation)

In [ ]:
fig1 = plt.figure()
ax = fig1.add_subplot(111, projection = '3d')
ax.set_box_aspect([1,1,1])
ax.scatter(correlated_variables[0, :], correlated_variables[1, :], correlated_variables[2, :])
ax.scatter(independent_variables[0, :], independent_variables[1, :], independent_variables[2, :])
ax.set_xlabel("Width")
ax.set_ylabel("Depth")
ax.set_zlabel("Height")
equal_scales(ax, correlated_variables)
plt.show()

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tinkerersnotes/tinkerersnotes/blob/main/statistics_playground.ipynb)

In [ ]:
covariance = np.cov(correlated_variables)
print("Covariance matrix of correlated variables:\n", covariance)

eigenvalues, eigenvectors = np.linalg.eigh(covariance)

# Sort from largest to smallest
idx = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues = eigenvalues[idx]
sorted_eigenvectors = eigenvectors[:, idx]

print("\nEigenvalues of covariance:\n", sorted_eigenvalues)
print("\nEigenvectors (as columns) of covariance:\n", sorted_eigenvectors)

print("\nStill covariance?\n", sorted_eigenvectors @ np.diag(sorted_eigenvalues) @ np.linalg.inv(sorted_eigenvectors))

# Note that Q^T will be Q^(-1)
print("\nOrthonormal basis if Q Q^T = I:\n", sorted_eigenvectors @ sorted_eigenvectors.T)

data_in_eigenvector_basis = sorted_eigenvectors.T @ correlated_variables
actual_correlation = np.corrcoef(data_in_eigenvector_basis)
print("\nNo linear correlation between orthogonal vectors:\n", actual_correlation)

fig2 = plt.figure()
ax = fig2.add_subplot(111, projection = '3d')
ax.set_box_aspect([1,1,1])
ax.scatter(data_in_eigenvector_basis[0, :], data_in_eigenvector_basis[1, :], data_in_eigenvector_basis[2, :])
# The new axes are the eigenvectors, largest variance first
for set_label, i in zip((ax.set_xlabel, ax.set_ylabel, ax.set_zlabel), range(3)):
    set_label(f"Principal axis {i + 1} (variance {sorted_eigenvalues[i]:.2f})")
equal_scales(ax, data_in_eigenvector_basis)
plt.show()
